# 투구 제구 성공 확률 모델 V3 학습

Direct Success, 실패 원인 분해, Trackman cohort mechanics, latent pitch type, robust Brier stacking을 하나의 leakage-safe 파이프라인으로 학습합니다. V2.2는 보존하고 V3 결과는 `runs/v3`에 따로 저장합니다.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

required_packages = {
    "catboost": ("catboost", "1.2.10"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "pyarrow": ("pyarrow", "25.0.1"),
    "scipy": ("scipy", "1.18.0"),
}
install = []
for module, (distribution, wanted) in required_packages.items():
    found = importlib.util.find_spec(module) is not None
    current = importlib.metadata.version(distribution) if found else None
    if current != wanted:
        install.append(f"{distribution}=={wanted}")
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])


In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from train_v3 import run_training

ROOT = Path.cwd().resolve()
if not (ROOT / "result.py").exists():
    raise FileNotFoundError("프로젝트 루트에서 main.ipynb를 실행해 주세요.")
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT / "runs" / "v3")).resolve()
print(f"project={ROOT}, run_dir={RUN_DIR}")
print(f"task_type={os.environ.get('BASEBALL_TASK_TYPE', 'GPU')}, devices={os.environ.get('BASEBALL_GPU_DEVICES', '0')}")


project=D:\baseball, run_dir=D:\baseball\runs\v3
task_type=GPU, devices=0


## V3 통합 학습 및 OOF 평가

기본값은 전체 데이터와 GPU 0번입니다. 빠른 점검만 할 때는 실행 전에 `BASEBALL_FAST_MODE=1`을 설정하세요. FAST_MODE 결과는 제출에 사용하면 안 됩니다.

In [3]:
result = run_training(ROOT, RUN_DIR)
display(result["blend_candidates"])
display(result["calibration_metrics"])
display(result["failure_metrics"])
print("Selected sources:", result["summary"]["ensemble"]["weight_map"])
print("Calibration:", result["summary"]["calibration"])
summary_keys = ["recent_weighted_cv", "brier_2024", "brier_2024_r", "brier_2024_f", "auc_2024"]
display(pd.Series({key: result["summary"][key] for key in summary_keys}, name="V3"))
print("Acceptance:", result["summary"]["acceptance"], "all_passed=", result["summary"]["all_acceptance_passed"])


Loading Trackman history
0:	learn: 1.3309385	total: 32.6ms	remaining: 4.53s
100:	learn: 0.8937906	total: 1.42s	remaining: 548ms
139:	learn: 0.8894440	total: 1.95s	remaining: 0us
0:	learn: 1.3351167	total: 19.5ms	remaining: 2.71s
100:	learn: 0.9096680	total: 1.71s	remaining: 660ms
139:	learn: 0.9045957	total: 3.85s	remaining: 0us
0:	learn: 1.3367204	total: 21.8ms	remaining: 3.03s
100:	learn: 0.9158181	total: 1.95s	remaining: 753ms
139:	learn: 0.9108593	total: 2.68s	remaining: 0us
0:	learn: 1.3380217	total: 24.2ms	remaining: 3.37s
100:	learn: 0.9209968	total: 2.19s	remaining: 847ms
139:	learn: 0.9168776	total: 3.02s	remaining: 0us


Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495552	test: 0.2494742	best: 0.2494742 (0)	total: 204ms	remaining: 2m 22s
100:	learn: 0.2428106	test: 0.2434450	best: 0.2434450 (100)	total: 12.9s	remaining: 1m 16s
200:	learn: 0.2417748	test: 0.2434908	best: 0.2434341 (148)	total: 24.7s	remaining: 1m 1s
bestTest = 0.243434086
bestIteration = 148
Shrink model to first 149 iterations.
0:	learn: 0.6778783	test: 0.6794636	best: 0.6794636 (0)	total: 199ms	remaining: 2m 19s
100:	learn: 0.5078759	test: 0.5246644	best: 0.5246644 (100)	total: 12.2s	remaining: 1m 12s
200:	learn: 0.5032805	test: 0.5248777	best: 0.5243263 (127)	total: 24.5s	remaining: 1m
bestTest = 0.5243263437
bestIteration = 127
Shrink model to first 128 iterations.
0:	learn: 0.6677634	test: 0.6698372	best: 0.6698372 (0)	total: 208ms	remaining: 2m 25s
100:	learn: 0.4028742	test: 0.4205899	best: 0.4205560 (84)	total: 12.2s	remaining: 1m 12s
bestTest = 0.4205559523
bestIteration = 84
Shrink model to first 85 iterations.
0:	learn: 0.6656636	test: 0.6631851	best: 0.663

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495285	test: 0.2499824	best: 0.2499824 (0)	total: 224ms	remaining: 2m 36s
bestTest = 0.249976978
bestIteration = 1
Shrink model to first 2 iterations.
0:	learn: 0.6777168	test: 0.6813116	best: 0.6813116 (0)	total: 133ms	remaining: 1m 32s
100:	learn: 0.5113872	test: 0.5856704	best: 0.5784691 (38)	total: 15s	remaining: 1m 29s
bestTest = 0.5784690841
bestIteration = 38
Shrink model to first 39 iterations.
0:	learn: 0.6685276	test: 0.6693661	best: 0.6693661 (0)	total: 150ms	remaining: 1m 44s
100:	learn: 0.4105454	test: 0.4276189	best: 0.4275794 (91)	total: 15.3s	remaining: 1m 30s
bestTest = 0.4275793873
bestIteration = 91
Shrink model to first 92 iterations.
0:	learn: 0.6649102	test: 0.6632483	best: 0.6632483 (0)	total: 248ms	remaining: 2m 53s
100:	learn: 0.3785366	test: 0.3695414	best: 0.3693727 (97)	total: 16.7s	remaining: 1m 39s
bestTest = 0.3693726602
bestIteration = 97
Shrink model to first 98 iterations.
0:	learn: 0.6307605	test: 0.6324424	best: 0.6324424 (0)	total: 279m

...

,candidate,brier_2022,brier_2022_r,brier_2022_f,brier_2023,brier_2023_r,brier_2023_f,brier_2024,brier_2024_r,brier_2024_f,score_primary,weighted_primary
0,identity_eta_0.00,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
1,identity_eta_0.25,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
2,identity_eta_0.50,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
3,identity_eta_0.75,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
4,identity_eta_1.00,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
5,affine_eta_0.00,0.243434,0.248628,0.206411,0.249945,0.249836,0.250878,0.248039,0.248119,0.247439,0.249230,0.247920
6,affine_eta_0.25,0.243434,0.248628,0.206411,0.249942,0.249851,0.250716,0.248012,0.248098,0.247365,0.249118,0.247904
7,affine_eta_0.50,0.243434,0.248628,0.206411,0.249949,0.249877,0.250562,0.247990,0.248083,0.247297,0.249026,0.247894
8,affine_eta_0.75,0.243434,0.248628,0.206411,0.249967,0.249914,0.250415,0.247974,0.248073,0.247237,0.248954,0.247891
9,affine_eta_1.00,0.243434,0.248628,0.206411,0.249996,0.249963,0.250278,0.247964,0.248068,0.247185,0.248903,0.247894


,season,target,rows,brier,target_mean,prediction_mean,best_iteration
0,2022,reverse,246994,0.172827,0.232945,0.236147,128
1,2022,middle,246994,0.126801,0.149400,0.152121,85
2,2022,outside_only,246994,0.108084,0.125505,0.125603,93
3,2022,reverse_middle,246994,0.035062,0.036560,0.038947,135
4,2023,reverse,245080,0.193138,0.255863,0.279572,39
5,2023,middle,245080,0.129597,0.153289,0.159185,92
6,2023,outside_only,245080,0.107678,0.125143,0.127187,98
7,2023,reverse_middle,245080,0.033337,0.034226,0.042579,145
8,2024,reverse,253035,0.184845,0.248001,0.254705,196
9,2024,middle,253035,0.144225,0.176292,0.172586,61


Selected sources: {'direct': 0.4157914390990381, 'structured_formula': 0.13854252622908414, 'structured_logit': 0.0, 'physics': 0.0, 'futures_anchor': 0.4456660346718777}
Calibration: {'version': 5, 'method': 'affine', 'slope': 1.155742094795486, 'intercept': -0.08150429990084007, 'eta': 1.0, 'selected_candidate': 'affine_eta_1.00', 'walk_forward_folds': [{'season': 2022, 'method': 'identity', 'eta': 0.0}, {'season': 2023, 'method': 'affine', 'slope': 1.104214185186829, 'intercept': -0.06165271776890524, 'eta': 1.0}, {'season': 2024, 'method': 'affine', 'slope': 1.0836547897693658, 'intercept': -0.04723675020996354, 'eta': 1.0}]}


recent_weighted_cv    0.247894
brier_2024            0.247964
brier_2024_r          0.248068
brier_2024_f          0.247185
auc_2024              0.547915
Name: V3, dtype: float64

Acceptance: {'recent_weighted_cv': False, 'brier_2024': False, 'brier_2024_r': False, 'brier_2024_f': False, 'auc_2024': False} all_passed= False


In [4]:
display(result["importance"].head(25))
display(pd.read_csv(result["submission_path"]))
print("다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.")


,feature,direct_success,reverse_expert,middle_expert,outside_expert,overlap_expert,physics_command,futures_expert,mean_importance
111,game_type,22.070491,2.464589,0.928662,0.178156,1.299393,16.350882,0.000000,6.184596
49,eb_reverse_pitcher_batter_hand,0.000000,16.512167,0.192169,0.803274,1.198112,8.898500,8.119105,5.103332
18,balls_before,0.311393,6.492065,4.032872,14.355502,0.918239,2.049022,0.000000,4.022728
182,season,24.810751,0.412849,1.851284,0.000000,0.091672,0.000000,0.000000,3.880937
27,batter_team_id,4.169319,0.515036,11.272444,1.285889,6.343632,0.000000,0.279824,3.409449
179,same_hand,4.081074,4.320327,0.143939,2.730842,1.378611,0.000000,9.375820,3.147230
119,is_two_strike,0.299867,3.576258,3.114155,13.440121,0.607133,0.000000,0.000000,3.005362
29,count_state_code,0.417718,7.201237,3.406986,6.315258,3.428573,0.000000,0.183327,2.993300
165,pitcher_team_id,3.176433,0.664554,9.469727,0.650484,6.581240,0.000000,0.000000,2.934634
184,strikes_before,0.082816,4.503218,3.186855,10.837329,0.493621,1.014543,0.263966,2.911764


,row_id,control_success
0,TEST_000001,0.442572
1,TEST_000017,0.443796
2,TEST_000213,0.463594
3,TEST_005332,0.501312
4,TEST_035185,0.482922


다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.
